##Entrega 3: Julia Tavares dos Santos

1. Configurando o ambiente utilizando SQLAlchemy
  instalação do driver de conexão e as bibliotecas, estabelecimento da conexão com o arquivo de banco de dados que será gerado localmente.

In [6]:
!pip install sqlalchemy psycopg2-binary pandas

import sqlalchemy
from sqlalchemy import create_engine, text
import pandas as pd

engine = create_engine('sqlite:///copa_do_mundo.db')
print("Conexão estabelecida.")

Conexão estabelecida.


2. Criando tabelas simples
Definição da estrutura relacional, a tabela paises armazena as seleções, e a tabela partidas armazena os jogos, utilizando uma chave estrangeira para garantir que uma partida só ocorra entre países cadastrados

In [11]:
with engine.connect() as conn:
    # permitir rerun
    conn.execute(text("DROP TABLE IF EXISTS partidas"))
    conn.execute(text("DROP TABLE IF EXISTS paises"))

    conn.execute(text("""
        CREATE TABLE paises (
            pais_id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL UNIQUE,
            continente TEXT
        )
    """))

    conn.execute(text("""
        CREATE TABLE partidas (
            partida_id INTEGER PRIMARY KEY AUTOINCREMENT,
            id_mandante INTEGER,
            id_visitante INTEGER,
            gols_mandante INTEGER,
            gols_visitante INTEGER,
            fase TEXT,
            FOREIGN KEY (id_mandante) REFERENCES paises(pais_id),
            FOREIGN KEY (id_visitante) REFERENCES paises(pais_id)
        )
    """))
    conn.commit()
    print("Tabelas recriadas.")

Tabelas recriadas.


3. Inserindo dados com Python

Alimentando banco de dados com informações: cadastro das seleções e registro dos resultados

In [12]:
with engine.connect() as conn:

    selecoes = [
        ('Brasil', 'América do Sul'), ('Croácia', 'Europa'),
        ('Argentina', 'América do Sul'), ('Holanda', 'Europa'),
        ('Marrocos', 'África'), ('Portugal', 'Europa'),
        ('Inglaterra', 'Europa'), ('França', 'Europa')
    ]
    for nome, cont in selecoes:
        conn.execute(text("INSERT INTO paises (nome, continente) VALUES (:n, :c)"), {"n": nome, "c": cont})

    jogos = [
        (2, 1, 1, 1, 'Quartas'), # Croácia 1x1 Brasil
        (4, 3, 2, 2, 'Quartas'), # Holanda 2x2 Argentina
        (5, 6, 1, 0, 'Quartas'), # Marrocos 1x0 Portugal
        (7, 8, 1, 2, 'Quartas'), # Inglaterra 1x2 França
        (3, 2, 3, 0, 'Semifinal'), # Argentina 3x0 Croácia
        (8, 5, 2, 0, 'Semifinal'), # França 2x0 Marrocos
        (3, 8, 3, 3, 'Final')      # Argentina 3x3 França
    ]
    for m, v, gm, gv, fase in jogos:
        conn.execute(text("""
            INSERT INTO partidas (id_mandante, id_visitante, gols_mandante, gols_visitante, fase)
            VALUES (:m, :v, :gm, :gv, :fase)
        """), {"m": m, "v": v, "gm": gm, "gv": gv, "fase": fase})

    conn.commit()
    print("Dados da copa 2022 carregados.")

Dados da Copa 2022 carregados.


4. Consulta 1: Dados entre partidas e nomes dos países

Quem jogou contra quem e qual foi o placar final.

In [13]:
query_confrontos = """
    SELECT
        p1.nome AS Mandante,
        pt.gols_mandante AS Gols_M,
        pt.gols_visitante AS Gols_V,
        p2.nome AS Visitante,
        pt.fase AS Torneio
    FROM partidas pt
    JOIN paises p1 ON pt.id_mandante = p1.pais_id
    JOIN paises p2 ON pt.id_visitante = p2.pais_id
"""

df_partidas = pd.read_sql(query_confrontos, engine)
df_partidas

,Mandante,Gols_M,Gols_V,Visitante,Torneio
0,Croácia,1,1,Brasil,Quartas
1,Holanda,2,2,Argentina,Quartas
2,Marrocos,1,0,Portugal,Quartas
3,Inglaterra,1,2,França,Quartas
4,Argentina,3,0,Croácia,Semifinal
5,França,2,0,Marrocos,Semifinal
6,Argentina,3,3,França,Final


4. Consulta 2: Média de gols por continente

Qual continente teve o melhor desempenho considerando a origem das seleções mandantes nas partidas registradas.

In [14]:
query_gols_continente = """
    SELECT
        p.continente,
        SUM(pt.gols_mandante + pt.gols_visitante) AS total_gols_partidas,
        COUNT(pt.partida_id) AS total_jogos
    FROM paises p
    JOIN partidas pt ON p.pais_id = pt.id_mandante
    GROUP BY p.continente
"""

df_continente = pd.read_sql(query_gols_continente, engine)
df_continente

,continente,total_gols_partidas,total_jogos
0,América do Sul,9,2
1,Europa,11,4
2,África,1,1


5. Conclusao

Como criar um banco de dados e realizar a conexão com Python, organizando as informações em tabelas separadas, inserindo dados e extraindo relatórios usando comandos SQL para cruzar e agrupar os valores salvos. Todas as etapas foram realizadas dentro do ambiente de nuvem